# 00 — Extracción Local de Tripletas Clínicas (PDAC) con Ollama

Este notebook extrae tripletas estructuradas `(sujeto, tipo_sujeto, relacion, objeto, tipo_objeto, score)` desde los documentos Markdown de `data/02_interim/` y guarda los JSON resultantes en `data/03_processed/` para alimentar el pipeline GraphRAG.

**Mejoras clave incluidas:**
- **Chunking consciente de Markdown:** preserva la jerarquía de títulos (`[Contexto: Documento > Sección]`) en cada fragmento y omite secciones de ruido (bibliografía/referencias).
- **System prompt clínico bilingüe (ES/EN) con ejemplos *few-shot*:** define explícitamente tipos de entidad, relaciones permitidas, dirección exacta y evita errores típicos (como confundir fármacos *comparados* con *combinados*).
- **Validación ontológica y deduplicación estricta:** descarta tripletas invertidas, genéricas o clínicamente incoherentes (p. ej. `DRUG TREATS PROCEDURE` o `REGIMEN COMBINES_WITH REGIMEN`).
- **Reintento automático y recuperación de JSON:** evita devoluciones vacías (`0 tripletas`) y rescata tripletas válidas si un fragmento se trunca.

### Instalación de Ollama

Instala la librería necesaria para conectar con el servicio local de Ollama.

In [ ]:
%pip install -q ollama

### Configuración y conexión

Configura las rutas, selecciona el modelo local y comprueba que Ollama está disponible.

In [8]:
import os
import re
import json
import glob
import time
import unicodedata
from collections import Counter
from ollama import Client

# Configuración de rutas
INTERIM_DIR = "../data/02_interim/"
PROCESSED_DIR = "../data/03_processed/"

# Inicializar cliente local de Ollama
OLLAMA_HOST = "http://localhost:11434"
MODEL_NAME = "qwen2.5:3b"
client = Client(host=OLLAMA_HOST)

# Verificar conexión y disponibilidad del modelo
try:
    available_models = [m.model for m in client.list().models]
    print(f"Conexión con Ollama OK ({OLLAMA_HOST})")
    print(f"Modelos disponibles: {available_models}")
    if MODEL_NAME not in available_models:
        print(f"[AVISO] '{MODEL_NAME}' no aparece en la lista. Ejecuta: ollama pull {MODEL_NAME}")
except Exception as e:
    print(f"[ERROR] No se pudo conectar con Ollama en {OLLAMA_HOST}: {e}")

Conexión con Ollama OK (http://localhost:11434)
Modelos disponibles: ['qwen2.5:3b', 'llama3.2:3b', 'nomic-embed-text:latest']


### Esquema de las tripletas

Define la estructura JSON, los tipos de entidades y las relaciones permitidas.

In [9]:
# Esquema JSON estricto que Ollama forzará a nivel de tokens
TRIPLET_SCHEMA = {
    "type": "object",
    "properties": {
        "tripletas": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "sujeto": {"type": "string"},
                    "tipo_sujeto": {
                        "type": "string",
                        "enum": ["DRUG", "REGIMEN", "DISEASE", "MUTATION", "STAGE", "ADVERSE_EVENT", "PROCEDURE", "TRIAL"]
                    },
                    "relacion": {
                        "type": "string",
                        "enum": ["TREATS", "COMBINES_WITH", "CAUSES", "CONTRAINDICATED_IN", "ASSOCIATED_WITH", "TARGETS", "EVALUATED_IN"]
                    },
                    "objeto": {"type": "string"},
                    "tipo_objeto": {
                        "type": "string",
                        "enum": ["DRUG", "REGIMEN", "DISEASE", "MUTATION", "STAGE", "ADVERSE_EVENT", "PROCEDURE", "TRIAL"]
                    },
                    "score": {"type": "number"}
                },
                "required": ["sujeto", "tipo_sujeto", "relacion", "objeto", "tipo_objeto", "score"]
            }
        }
    },
    "required": ["tripletas"]
}

SYSTEM_PROMPT = """You are an expert clinical oncology knowledge graph extractor specializing in Pancreatic Ductal Adenocarcinoma (PDAC) and oncology pharmacology.
Extract factual, atomic medical triplets explicitly stated in the provided text fragment and return ONLY valid JSON matching the schema.

### ENTITY TYPES:
- DRUG: Individual pharmaceutical agent (e.g., "Gemcitabina", "Cisplatino", "Oxaliplatin", "Irinotecan", "Nab-paclitaxel", "5-Fluorouracil").
- REGIMEN: Multi-drug protocol or named combination (e.g., "FOLFIRINOX", "mFOLFIRINOX", "NALIRIFOX", "GEM-NABP", "NABPLAGEM").
- DISEASE: Specific cancer or clinical condition (e.g., "Adenocarcinoma de páncreas", "Pancreatic ductal adenocarcinoma", "Cáncer de vejiga", "Diabetes").
- MUTATION: Gene, mutation, or biomarker (e.g., "KRAS", "BRCA1", "BRCA2", "PALB2", "TP53", "CA19-9", "UGT1A1").
- STAGE: Cancer stage or resectability status (e.g., "Metastatic", "Locally advanced", "Resectable", "Borderline resectable", "Stage IV").
- ADVERSE_EVENT: Specific toxicity, side effect, or clinical complication (e.g., "Neutropenia", "Trombocitopenia", "Anemia", "Diarrea", "Neuropatía periférica", "Neumonitis"). Prefer specific toxicities over vague terms.
- PROCEDURE: Diagnostic, surgical, or therapeutic intervention (e.g., "Radioterapia", "Surgical resection", "MRI", "CT scan", "EUS-FNA").
- TRIAL: Clinical trial ID or name (e.g., "NCT06115499", "PLATINUM Trial", "NAPOLI 3", "PRODIGE 24", "ESPAC-4").

### ALLOWED RELATIONS AND STRICT DIRECTION (sujeto -> relacion -> objeto):
- TREATS: DRUG / REGIMEN / PROCEDURE -> DISEASE / STAGE
- COMBINES_WITH: DRUG -> DRUG, or DRUG / REGIMEN -> PROCEDURE (ONLY when administered TOGETHER in the same treatment).
- CAUSES: DRUG / REGIMEN / PROCEDURE / DISEASE / MUTATION -> ADVERSE_EVENT / DISEASE
- CONTRAINDICATED_IN: DRUG / REGIMEN / PROCEDURE -> DISEASE / ADVERSE_EVENT / STAGE / PROCEDURE / DRUG
- TARGETS: DRUG / REGIMEN -> MUTATION / DISEASE
- EVALUATED_IN: DRUG / REGIMEN / PROCEDURE -> TRIAL
- ASSOCIATED_WITH: MUTATION / DISEASE / STAGE / ADVERSE_EVENT -> DISEASE / STAGE / MUTATION / ADVERSE_EVENT

### STRICT QUALITY RULES:
1. GROUNDING RULE: Extract ONLY clinical relationships explicitly stated in the text fragment. Every sujeto and objeto MUST appear in the text fragment or its [Contexto] header. NEVER invent diseases, genes, or drugs from outside memory.
2. NEVER invert relation direction (a DISEASE never TREATS a DRUG; a DRUG never TREATS a PROCEDURE or ADVERSE_EVENT).
3. NEVER use COMBINES_WITH for treatments/regimens being COMPARED against each other (e.g., "A vs B", "A compared with B", "A frente a B"). COMBINES_WITH is ONLY for co-administered therapies.
4. Use concise canonical entity names (1 to 6 words). NEVER use full sentences or generic placeholders ("patients", "pacientes", "treatment", "tratamiento", "study", "estudio", "drug", "fármaco", "toxicity", "toxicidad", "cancer", "cáncer").
5. Always use the [Contexto] header to resolve which drug, regimen, or trial the fragment refers to when omitted in bullet lists or tables.
6. Assign `score` between 0.70 and 1.00 reflecting how explicitly the relationship is stated in the text.
7. If the fragment only contains packaging/metadata or no clinical facts, return {"tripletas": []}.

### EXAMPLES:
Example 1 (Spanish):
[Contexto: Gemcitabina > 4.1. Indicaciones terapéuticas]
Gemcitabina está indicada en adenocarcinoma de páncreas metastásico. En combinación con cisplatino trata el cáncer de vejiga. Está contraindicada en lactancia y causa neutropenia.
Output:
{"tripletas": [
  {"sujeto": "Gemcitabina", "tipo_sujeto": "DRUG", "relacion": "TREATS", "objeto": "Adenocarcinoma de páncreas", "tipo_objeto": "DISEASE", "score": 0.98},
  {"sujeto": "Gemcitabina", "tipo_sujeto": "DRUG", "relacion": "COMBINES_WITH", "objeto": "Cisplatino", "tipo_objeto": "DRUG", "score": 0.95},
  {"sujeto": "Gemcitabina", "tipo_sujeto": "DRUG", "relacion": "TREATS", "objeto": "Cáncer de vejiga", "tipo_objeto": "DISEASE", "score": 0.95},
  {"sujeto": "Gemcitabina", "tipo_sujeto": "DRUG", "relacion": "CONTRAINDICATED_IN", "objeto": "Lactancia", "tipo_objeto": "ADVERSE_EVENT", "score": 0.95},
  {"sujeto": "Gemcitabina", "tipo_sujeto": "DRUG", "relacion": "CAUSES", "objeto": "Neutropenia", "tipo_objeto": "ADVERSE_EVENT", "score": 0.95}
]}

Example 2 (English):
[Contexto: NCT06115499 > Brief Summary]
The PLATINUM trial (NCT06115499) compares NABPLAGEM versus Nab-paclitaxel/Gemcitabine in metastatic PDAC patients with BRCA1/2 or PALB2 mutations.
Output:
{"tripletas": [
  {"sujeto": "NABPLAGEM", "tipo_sujeto": "REGIMEN", "relacion": "EVALUATED_IN", "objeto": "NCT06115499", "tipo_objeto": "TRIAL", "score": 0.98},
  {"sujeto": "NABPLAGEM", "tipo_sujeto": "REGIMEN", "relacion": "TREATS", "objeto": "Pancreatic ductal adenocarcinoma", "tipo_objeto": "DISEASE", "score": 0.95},
  {"sujeto": "Nab-paclitaxel", "tipo_sujeto": "DRUG", "relacion": "COMBINES_WITH", "objeto": "Gemcitabine", "tipo_objeto": "DRUG", "score": 0.95},
  {"sujeto": "BRCA1/2", "tipo_sujeto": "MUTATION", "relacion": "ASSOCIATED_WITH", "objeto": "Pancreatic ductal adenocarcinoma", "tipo_objeto": "DISEASE", "score": 0.92},
  {"sujeto": "PALB2", "tipo_sujeto": "MUTATION", "relacion": "ASSOCIATED_WITH", "objeto": "Pancreatic ductal adenocarcinoma", "tipo_objeto": "DISEASE", "score": 0.92}
]}
"""

### División de documentos en fragmentos

Divide los documentos Markdown conservando el contexto de sus títulos y secciones.

In [10]:
SKIP_HEADINGS = {
    "references", "bibliography", "referencias", "bibliografia",
    "acknowledgements", "acknowledgments", "agradecimientos",
    "funding", "conflict of interest", "conflicts of interest",
    "declarations", "author contributions", "abbreviations"
}

def _normalize_key(text: str) -> str:
    text = str(text or "").strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"\s+", " ", text)
    return text

def split_markdown_into_chunks(text: str, doc_title: str = "", chunk_size: int = 2000, overlap: int = 150):
    """Divide un documento Markdown respetando encabezados y párrafos,
    anteponiendo el contexto jerárquico (# > ## > ###) a cada fragmento."""
    lines = text.splitlines()
    headings = {}
    sections = []
    current_lines = []
    skip_level = None

    def flush_section():
        nonlocal current_lines
        body = "\n".join(current_lines).strip()
        if body:
            path_parts = [headings[k] for k in sorted(headings.keys())]
            if not path_parts and doc_title:
                path_parts = [doc_title]
            context = " > ".join(path_parts)
            sections.append((context, body))
        current_lines = []

    for line in lines:
        m = re.match(r"^(#{1,4})\s+(.+)$", line.strip())
        if m:
            flush_section()
            level = len(m.group(1))
            title = m.group(2).strip()
            norm_title = _normalize_key(title)
            if skip_level is not None and level <= skip_level:
                skip_level = None
            if norm_title in SKIP_HEADINGS:
                skip_level = level
                continue
            if skip_level is not None:
                continue
            headings[level] = title
            for k in list(headings.keys()):
                if k > level:
                    del headings[k]
        else:
            if skip_level is None:
                current_lines.append(line)
    flush_section()

    chunks = []
    for context, body in sections:
        header = f"[Contexto: {context}]\n" if context else ""
        max_body = max(500, chunk_size - len(header))
        if len(body) <= max_body:
            chunks.append(header + body)
        else:
            paragraphs = [p.strip() for p in re.split(r"\n+", body) if p.strip()]
            buf = ""
            for p in paragraphs:
                if len(buf) + len(p) + 1 <= max_body:
                    buf = (buf + "\n" + p).strip()
                else:
                    if buf:
                        chunks.append(header + buf)
                        tail = buf[-overlap:] if overlap > 0 and len(buf) > overlap else ""
                        buf = (tail + "\n" + p).strip() if tail else p
                    else:
                        step = max(200, max_body - overlap)
                        for i in range(0, len(p), step):
                            chunks.append(header + p[i:i + max_body])
                        buf = ""
            if buf:
                chunks.append(header + buf)
    return chunks

### Validación y limpieza

Filtra tripletas inválidas, comprueba su presencia en el texto y elimina duplicados.

In [11]:
# Reglas ontológicas estrictas: relacion -> (tipos_sujeto_permitidos, tipos_objeto_permitidos)
VALID_ONTOLOGY_RULES = {
    "TREATS": ({"DRUG", "REGIMEN", "PROCEDURE"}, {"DISEASE", "STAGE"}),
    "COMBINES_WITH": ({"DRUG", "REGIMEN", "PROCEDURE"}, {"DRUG", "PROCEDURE"}),
    "CAUSES": ({"DRUG", "REGIMEN", "PROCEDURE", "DISEASE", "MUTATION"}, {"ADVERSE_EVENT", "DISEASE"}),
    "CONTRAINDICATED_IN": ({"DRUG", "REGIMEN", "PROCEDURE"}, {"DISEASE", "ADVERSE_EVENT", "STAGE", "PROCEDURE", "DRUG"}),
    "TARGETS": ({"DRUG", "REGIMEN"}, {"MUTATION", "DISEASE"}),
    "EVALUATED_IN": ({"DRUG", "REGIMEN", "PROCEDURE"}, {"TRIAL"}),
    "ASSOCIATED_WITH": ({"MUTATION", "DISEASE", "STAGE", "ADVERSE_EVENT"}, {"DISEASE", "STAGE", "MUTATION", "ADVERSE_EVENT"}),
}

# Entidades demasiado genéricas que ensucian el grafo de conocimiento
GENERIC_STOPWORDS = {
    "patients", "pacientes", "patient", "paciente",
    "treatment", "tratamiento", "therapy", "terapia", "therapies", "terapias",
    "drug", "drugs", "farmaco", "farmacos", "medicamento", "medicamentos", "agente", "agentes",
    "study", "estudio", "trial", "ensayo", "ensayo clinico", "clinical trial", "clinical trials",
    "disease", "enfermedad", "cancer", "tumor", "tumores", "tumors", "neoplasm", "neoplasia",
    "toxicity", "toxicidad", "toxicidades", "toxicities", "toxic effects",
    "adverse event", "adverse events", "reacciones adversas", "reaccion adversa",
    "efectos adversos", "efectos secundarios", "side effects", "sintomas", "symptoms",
    "chemotherapy", "quimioterapia", "regimen", "regimenes", "regimens",
    "arm", "brazo", "group", "grupo", "placebo", "control", "monotherapy", "monoterapia",
    "combination", "combinacion", "standard of care", "unknown", "none", "n/a", "other", "otros"
}

def _clean_entity(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    text = text.strip(".,;:-–—\"'[]*`")
    return text

GROUNDING_IGNORE_TOKENS = {
    "de", "del", "la", "las", "el", "los", "en", "con", "por", "para", "sin", "al", "un", "una", "y", "o",
    "of", "the", "and", "or", "for", "with", "in", "to", "on", "at", "by", "non", "no",
    "cancer", "carcinoma", "adenocarcinoma", "tumor", "sindrome", "syndrome", "disease", "enfermedad",
    "trastorno", "disorder", "grado", "grade", "tipo", "type", "cell", "celulas", "severe", "severa", "grave",
    "acute", "aguda", "chronic", "cronica", "advanced", "avanzado", "localmente", "locally", "metastatic", "metastasico"
}

def _is_grounded_in_chunk(entity: str, norm_chunk: str) -> bool:
    """Verifica que la entidad aparezca realmente en el fragmento origen para bloquear alucinaciones."""
    norm_ent = _normalize_key(entity)
    if not norm_ent:
        return False
    if norm_ent in norm_chunk:
        return True
    # Equivalencia común PDAC <-> pancreas / pancreatic
    if norm_ent == "pdac" and ("pancrea" in norm_chunk):
        return True
    if "pancreatic ductal adenocarcinoma" in norm_ent and ("pdac" in norm_chunk or "pancrea" in norm_chunk):
        return True
    tokens = [tok for tok in re.findall(r"[a-z0-9]+", norm_ent) if len(tok) >= 2 and tok not in GROUNDING_IGNORE_TOKENS]
    if not tokens:
        return norm_ent in norm_chunk
    for tok in tokens:
        stem = tok[:6] if len(tok) >= 6 else tok
        if stem not in norm_chunk:
            return False
    return True

def validate_and_clean_triplets(raw_triplets, min_score: float = 0.65, source_chunk: str = None):
    """Filtra tripletas inválidas según la ontología clínica, verifica grounding en el texto y deduplica."""
    valid = {}
    rejected = 0
    norm_chunk = _normalize_key(source_chunk) if source_chunk else None

    for t in raw_triplets:
        suj = _clean_entity(t.get("sujeto", ""))
        obj = _clean_entity(t.get("objeto", ""))
        t_suj = str(t.get("tipo_sujeto", "")).strip().upper()
        t_obj = str(t.get("tipo_objeto", "")).strip().upper()
        rel = str(t.get("relacion", "")).strip().upper()

        try:
            score = round(min(1.0, max(0.0, float(t.get("score", 0.0)))), 2)
        except (TypeError, ValueError):
            score = 0.0

        norm_suj = _normalize_key(suj)
        norm_obj = _normalize_key(obj)

        # Validaciones de higiene básica
        if not suj or not obj or len(suj) < 2 or len(obj) < 2:
            rejected += 1
            continue
        if len(suj) > 85 or len(obj) > 85 or len(suj.split()) > 10 or len(obj.split()) > 10:
            rejected += 1
            continue
        if norm_suj == norm_obj:
            rejected += 1
            continue
        if norm_suj in GENERIC_STOPWORDS or norm_obj in GENERIC_STOPWORDS:
            rejected += 1
            continue
        if score < min_score:
            rejected += 1
            continue

        # Verificación de grounding en el texto origen (bloquea alucinaciones de memoria)
        if norm_chunk is not None:
            if not _is_grounded_in_chunk(suj, norm_chunk) or not _is_grounded_in_chunk(obj, norm_chunk):
                rejected += 1
                continue

        # Validación ontológica de dominio/rango
        if rel not in VALID_ONTOLOGY_RULES:
            rejected += 1
            continue
        allowed_suj, allowed_obj = VALID_ONTOLOGY_RULES[rel]
        if t_suj not in allowed_suj or t_obj not in allowed_obj:
            rejected += 1
            continue
        # Evitar falsos COMBINES_WITH entre dos regímenes distintos comparados en un ensayo
        if rel == "COMBINES_WITH" and t_suj == "REGIMEN" and t_obj == "REGIMEN":
            rejected += 1
            continue

        # Deduplicación conservando el mayor score
        key = (norm_suj, rel, norm_obj)
        clean_t = {
            "sujeto": suj,
            "tipo_sujeto": t_suj,
            "relacion": rel,
            "objeto": obj,
            "tipo_objeto": t_obj,
            "score": score
        }
        if key not in valid or score > valid[key]["score"]:
            valid[key] = clean_t

    return list(valid.values()), rejected

### Extracción de tripletas

Envía cada fragmento a Ollama, recupera el JSON y aplica las validaciones definidas.

In [12]:
def _call_ollama_chunk(chunk: str, temperature: float = 0.1, retry_hint: bool = False):
    user_msg = (
        "Extract ALL explicit factual clinical triplets (drugs, regimens, diseases, mutations, stages, "
        "adverse events, procedures, trials) from the following text fragment:\n\n" + chunk
    )
    if retry_hint:
        user_msg += (
            "\n\nIMPORTANT: Look carefully for any drug indications (TREATS), co-administered therapies "
            "(COMBINES_WITH), adverse events (CAUSES), contraindications (CONTRAINDICATED_IN), "
            "mutations/biomarkers (ASSOCIATED_WITH/TARGETS), or clinical trials (EVALUATED_IN) stated in this text."
        )

    response = client.chat(
        model=MODEL_NAME,
        format=TRIPLET_SCHEMA,
        options={"temperature": temperature, "num_ctx": 4096, "num_predict": 1536},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ]
    )
    content = response["message"]["content"]

    try:
        return json.loads(content).get("tripletas", [])
    except json.JSONDecodeError:
        # Rescate de tripletas completas si el JSON se truncó al final
        matches = re.findall(r'\{[^{}]*"sujeto"[^{}]*"score"[^{}]*\}', content)
        salvaged = []
        for m in matches:
            try:
                salvaged.append(json.loads(m))
            except Exception:
                pass
        return salvaged

def extract_triplets_from_text(full_text: str, doc_title: str = "", verbose: bool = False):
    chunks = split_markdown_into_chunks(full_text, doc_title=doc_title)
    grounded_triplets = []
    total_raw = 0
    total_rejected = 0

    for i, chunk in enumerate(chunks, 1):
        try:
            triplets = _call_ollama_chunk(chunk, temperature=0.1, retry_hint=False)
            # Reintento ligero si un fragmento largo informativo devuelve 0 tripletas
            if not triplets and len(chunk) > 350:
                triplets = _call_ollama_chunk(chunk, temperature=0.25, retry_hint=True)
            total_raw += len(triplets)
            # Validar ontología + grounding inmediatamente contra ESTE fragmento
            chunk_valid, chunk_rejected = validate_and_clean_triplets(triplets, source_chunk=chunk)
            total_rejected += chunk_rejected
            grounded_triplets.extend(chunk_valid)
            if verbose:
                print(f"    Fragmento {i}/{len(chunks)}: {len(triplets)} brutas -> {len(chunk_valid)} válidas ({chunk_rejected} descartadas)")
        except Exception as e:
            print(f"    [Aviso] Fragmento {i}/{len(chunks)} omitido: {e}")

    # Deduplicación global entre fragmentos del documento
    final_triplets, _ = validate_and_clean_triplets(grounded_triplets, source_chunk=None)
    if verbose:
        print(f"    -> Resumen documento: {total_raw} brutas -> {len(final_triplets)} válidas únicas ({total_rejected} descartadas por ontología/grounding)")

    return {"tripletas": final_triplets}

### Prueba sobre un documento

Ejecuta una extracción de prueba, muestra las tripletas válidas y resume sus relaciones.

In [13]:
# === PRUEBA UNITARIA ===
# Ejecuta primero esta celda para comprobar la calidad y velocidad sobre un solo documento
test_file = "../data/02_interim/04_farmacos/Gemcitabina.md"

print(f"Iniciando prueba unitaria sobre: {test_file}")
inicio = time.time()

with open(test_file, "r", encoding="utf-8") as f:
    sample_text = f.read()

doc_name = os.path.splitext(os.path.basename(test_file))[0]
resultado_prueba = extract_triplets_from_text(sample_text, doc_title=doc_name, verbose=True)
duracion = round(time.time() - inicio, 2)

tripletas = resultado_prueba["tripletas"]
print(f"\nCompletado en {duracion} segundos.")
print(f"Total tripletas válidas extraídas: {len(tripletas)}")
print("Distribución por relación:", dict(Counter(t["relacion"] for t in tripletas)))
print("\nMuestra de las primeras 15 tripletas:")
print(json.dumps(tripletas[:15], indent=2, ensure_ascii=False))

Iniciando prueba unitaria sobre: ../data/02_interim/04_farmacos/Gemcitabina.md
    Fragmento 1/20: 1 brutas -> 0 válidas (1 descartadas)
    Fragmento 2/20: 5 brutas -> 5 válidas (0 descartadas)
    Fragmento 3/20: 1 brutas -> 1 válidas (0 descartadas)
    Fragmento 4/20: 20 brutas -> 5 válidas (15 descartadas)
    Fragmento 5/20: 6 brutas -> 6 válidas (0 descartadas)
    Fragmento 6/20: 12 brutas -> 9 válidas (3 descartadas)
    Fragmento 7/20: 20 brutas -> 20 válidas (0 descartadas)
    Fragmento 8/20: 23 brutas -> 23 válidas (0 descartadas)
    Fragmento 9/20: 24 brutas -> 23 válidas (0 descartadas)
    Fragmento 10/20: 6 brutas -> 6 válidas (0 descartadas)
    Fragmento 11/20: 9 brutas -> 9 válidas (0 descartadas)
    Fragmento 12/20: 9 brutas -> 9 válidas (0 descartadas)
    Fragmento 13/20: 8 brutas -> 8 válidas (0 descartadas)
    Fragmento 14/20: 10 brutas -> 10 válidas (0 descartadas)
    Fragmento 15/20: 3 brutas -> 3 válidas (0 descartadas)
    Fragmento 16/20: 1 brutas -> 0

### Procesamiento completo

Procesa los documentos de entrada y guarda las tripletas resultantes en formato JSON.

In [14]:
# === PROCESAMIENTO COMPLETO DE TODOS LOS DOCUMENTOS ===
def process_all_documents(limpiar_previos: bool = False, sobrescribir: bool = False, verbose_chunks: bool = False):
    """Procesa todos los archivos .md en data/02_interim/ y guarda los JSON en data/03_processed/.
    - limpiar_previos=True: borra todos los JSON previos antes de empezar.
    - sobrescribir=False: permite reanudar si se interrumpe la ejecución saltando archivos ya generados.
    """
    if limpiar_previos:
        previos = glob.glob(os.path.join(PROCESSED_DIR, "**/*.json"), recursive=True)
        for f in previos:
            os.remove(f)
        print(f"Limpieza completada: {len(previos)} archivos previos eliminados.\n")

    subfolders = [f.path for f in os.scandir(INTERIM_DIR) if f.is_dir()]
    total_docs = 0
    total_triplets = 0
    inicio_global = time.time()

    for folder in sorted(subfolders):
        category_name = os.path.basename(folder)
        output_folder = os.path.join(PROCESSED_DIR, category_name)
        os.makedirs(output_folder, exist_ok=True)

        md_files = sorted(glob.glob(os.path.join(folder, "*.md")))
        print(f"=== Categoría: {category_name} ({len(md_files)} documentos) ===")

        for file_path in md_files:
            file_name = os.path.basename(file_path)
            doc_id = os.path.splitext(file_name)[0]
            output_file = os.path.join(output_folder, f"{doc_id}_triplets.json")

            if not sobrescribir and not limpiar_previos and os.path.exists(output_file):
                with open(output_file, "r", encoding="utf-8") as f:
                    existing = json.load(f)
                n_existing = len(existing.get("tripletas", []))
                if n_existing > 0:
                    print(f"  [Saltado] {file_name} ya procesado ({n_existing} tripletas)")
                    total_docs += 1
                    total_triplets += n_existing
                    continue

            t0 = time.time()
            print(f"  Procesando {file_name}...", flush=True)

            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            extracted_data = extract_triplets_from_text(text, doc_title=doc_id, verbose=verbose_chunks)
            triplets_list = extracted_data.get("tripletas", [])

            final_json = {
                "doc_id": doc_id,
                "categoria": category_name,
                "tripletas": triplets_list
            }

            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(final_json, f, indent=2, ensure_ascii=False)

            elapsed = round(time.time() - t0, 1)
            print(f"  -> OK {file_name}: {len(triplets_list)} tripletas en {elapsed}s")

            total_docs += 1
            total_triplets += len(triplets_list)

    tiempo_total = round((time.time() - inicio_global) / 60, 2)
    print(f"\nPROCESO FINALIZADO en {tiempo_total} minutos.")
    print(f"Total documentos procesados: {total_docs}")
    print(f"Total tripletas generadas: {total_triplets}")

# Lanzar el procesamiento completo (cambia limpiar_previos=True si quieres regenerar todo desde cero)
process_all_documents(limpiar_previos=False, sobrescribir=False, verbose_chunks=False)

=== Categoría: 01_guias (5 documentos) ===
  Procesando Article_SEOMClinicalGuidelinesForPancr.md...
  -> OK Article_SEOMClinicalGuidelinesForPancr.md: 257 tripletas en 838.4s
  Procesando ES-Cancer-de-Pancreas-Guia-para-Pacientes.md...
  -> OK ES-Cancer-de-Pancreas-Guia-para-Pacientes.md: 224 tripletas en 829.3s
  Procesando PIIS0923753423008244.md...
  -> OK PIIS0923753423008244.md: 201 tripletas en 783.0s
  Procesando pancreatic-patient-esl.md...
  -> OK pancreatic-patient-esl.md: 272 tripletas en 1500.4s
  Procesando pancreatic.md...
  -> OK pancreatic.md: 654 tripletas en 3654.1s
=== Categoría: 02_papers (9 documentos) ===
  Procesando 1-s2.0-S1368764623000432-main.md...
  -> OK 1-s2.0-S1368764623000432-main.md: 52 tripletas en 1771.4s
  Procesando 20191119-Evolucion-2013-18-de-la-superv-con-tto-adyuv-Cancer-Pancreas-1.md...
  -> OK 20191119-Evolucion-2013-18-de-la-superv-con-tto-adyuv-Cancer-Pancreas-1.md: 22 tripletas en 292.6s
  Procesando CAAC-76-0.md...
  -> OK CAAC-76-0.md: 